# A6000-Optimized Large-Scale Structure Formation MC

500M defects on 256³ grid — safe for 48 GB VRAM.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

# ====================== A6000 SAFE PARAMETERS ======================
N_defects = 500_000_000   # 500 million — safe for 48 GB VRAM
box_size = 2000.0         # Mpc
grid_size = 256           # 256³ — safe
defect_size = 8.0         # Mpc
seed = 42

print(f"Starting LARGE simulation: {N_defects:,} defects on {grid_size}³ grid")
start_time = time.time()

# ====================== GENERATE DEFECTS (CPU, float32) ======================
np.random.seed(seed)
positions = np.random.uniform(0, box_size, (N_defects, 3)).astype(np.float32)

# Clustering loop
for i in tqdm(range(20000), desc="Adding clustering"):
    idx = np.random.randint(0, N_defects)
    positions[idx] += np.random.normal(0, defect_size, 3).astype(np.float32)
    positions[idx] = np.clip(positions[idx], 0, box_size)

# Move to GPU
positions_gpu = cp.asarray(positions)

# ====================== DENSITY FIELD (GPU) ======================
print("Building density field on GPU...")
density_gpu = cp.zeros((grid_size, grid_size, grid_size), dtype=cp.float32)
indices = (positions_gpu / box_size * grid_size).astype(cp.int32)
cp.add.at(density_gpu, (indices[:,0], indices[:,1], indices[:,2]), 1.0)

# ====================== POWER SPECTRUM (GPU FFT) ======================
print("Computing power spectrum on GPU...")
density_ft = cp.fft.fftn(density_gpu)
power = cp.abs(density_ft)**2

# Radial average (CPU)
power_cpu = cp.asnumpy(power)
k = np.fft.fftfreq(grid_size, d=box_size/grid_size) * 2 * np.pi
k_mag = np.sqrt(k[:,None,None]**2 + k[None,:,None]**2 + k[None,None,:]**2)
k_bins = np.linspace(0, k.max(), 200)
p_k_1d = np.zeros(len(k_bins)-1)

for i in range(len(k_bins)-1):
    mask = (k_mag >= k_bins[i]) & (k_mag < k_bins[i+1])
    if mask.sum() > 0:
        p_k_1d[i] = power_cpu[mask].mean()

k_centers = (k_bins[:-1] + k_bins[1:]) / 2

print(f"Simulation completed in {time.time() - start_time:.1f} seconds")
print(f"Mean density: {cp.asnumpy(density_gpu.mean()):.4f}")

# Plot
plt.figure(figsize=(10,6))
plt.loglog(k_centers, p_k_1d, 'b-', label='P(k) from lattice defects')
plt.xlabel('k (Mpc^{-1})')
plt.ylabel('P(k)')
plt.title(f'Power Spectrum from {N_defects:,} Lattice Defects (A6000 GPU)')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.savefig('../figures/power-spectrum-500M-gpu.pdf')
plt.show()